# Assignment 4 — Active Learning Experiment

Этот ноутбук повторяет rubric задания 4:

- старт с `N=50`;
- 5 AL-итераций;
- сравнение `entropy`, `margin` и `random`;
- learning curves;
- оценка sample savings относительно `random`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from agents.active_learning_agent import ActiveLearningAgent

candidate_paths = [
    ROOT / "data/labeled/final_dataset.parquet",
    ROOT / "data/raw/merged_raw.csv",
]
for path in candidate_paths:
    if path.exists():
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
else:
    raise FileNotFoundError("No input dataset found in data/labeled/final_dataset.parquet or data/raw/merged_raw.csv")

if "final_label" in df.columns and "label" not in df.columns:
    df = df.rename(columns={"final_label": "label"})

df = df.dropna(subset=["text", "label"]).copy()
agent = ActiveLearningAgent(config=ROOT / "config.yaml")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
start_n = min(50, max(2, len(train_df) // 3))
labeled_seed, pool_seed = train_test_split(train_df, train_size=start_n, random_state=42)

print({"rows": len(df), "start_n": len(labeled_seed), "pool": len(pool_seed), "test": len(test_df)})

In [ ]:
strategies = ["entropy", "margin", "random"]
histories = {}

for strategy in strategies:
    histories[strategy] = agent.run_cycle(
        labeled_df=labeled_seed.copy(),
        pool_df=pool_seed.copy(),
        test_df=test_df.copy(),
        strategy=strategy,
        n_iterations=5,
        batch_size=min(20, len(pool_seed)),
    )

histories

In [ ]:
history_df = pd.concat(
    [pd.DataFrame(rows).assign(strategy=name) for name, rows in histories.items()],
    ignore_index=True,
)
history_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for strategy_name, sub in history_df.groupby("strategy"):
    axes[0].plot(sub["iteration"], sub["f1_macro"], marker="o", label=strategy_name)
    axes[1].plot(sub["iteration"], sub["accuracy"], marker="o", label=strategy_name)
axes[0].set_title("F1 macro by iteration")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("f1_macro")
axes[1].set_title("Accuracy by iteration")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("accuracy")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
target_f1 = history_df[history_df["strategy"] == "random"]["f1_macro"].max()
rows = []
for strategy_name, sub in history_df.groupby("strategy"):
    reached = sub[sub["f1_macro"] >= target_f1].sort_values("n_labeled")
    n_needed = None if reached.empty else int(reached.iloc[0]["n_labeled"])
    rows.append({"strategy": strategy_name, "target_f1": float(target_f1), "n_labeled_to_reach_target": n_needed})

savings_df = pd.DataFrame(rows)
random_n = savings_df.loc[savings_df["strategy"] == "random", "n_labeled_to_reach_target"].iloc[0]
savings_df["sample_savings_vs_random"] = savings_df["n_labeled_to_reach_target"].apply(
    lambda x: None if pd.isna(x) or pd.isna(random_n) else int(random_n - x)
)
savings_df

## Что показать на защите

- что `entropy` и `margin` достигают того же качества быстрее или не хуже `random`;
- что в history есть `n_labeled`, `accuracy`, `f1_macro`;
- что AL реально экономит ручную разметку по сравнению с random baseline.